# `verification.py` — Vérifie vos fonctions contre les tableaux du guide

> **MODULE FOURNI — NE PAS MODIFIER**

Relancez-le aussi souvent que vous voulez : dans le menu, **Exécution → Tout exécuter**
(`Run all`), ou cellule par cellule avec **Maj + Entrée**.

Chaque fonction que vous complétez fait passer une ligne de « `...` » à **OK**.
Les exemples utilisés sont **EXACTEMENT** ceux des tableaux d'observation du guide :
si vous les avez classés à la main, vous savez déjà quoi obtenir.

| Affichage | Sens |
|---|---|
| `OK` | la fonction renvoie la valeur attendue |
| `...` | la fonction n'est pas encore écrite (elle renvoie `None`) |
| `X` | la fonction répond, mais pas ce qu'il faut — ou elle plante |

**Une cellule à la fois :** chaque section est indépendante et peut être relancée seule.
Le bilan de la dernière cellule ne compte que les essais réellement exécutés :
pour un décompte complet, exécutez tout le carnet.

## Imports et compteur

`_stats` retient le nombre d'essais réussis, échoués et en attente.

> **Rappel Colab :** vous modifiez un `.py`, vous relancez, et rien ne change ?
> C'est l'ancienne version qui tourne. Les deux lignes `%load_ext autoreload`
> et `%autoreload 2` rechargent vos modules à chaque exécution.

In [ ]:
%load_ext autoreload
%autoreload 2

import phishing_rules as ph
import spam_rules as sp

_stats = {"ok": 0, "echec": 0, "attente": 0}

## Les trois outils de vérification

`essai(nom, fonction, attendu)` exécute votre fonction et compare le résultat.

Deux subtilités :

- `None` est parfois la **BONNE** réponse (par exemple « aucune marque détectée »).
  On ne le lit comme « pas encore fait » que si autre chose était attendu.
- Les nombres à virgule se comparent avec une **tolérance** : `0.1 + 0.2` ne
  vaut pas exactement `0.3` en Python.

`points(couple)` extrait les points d'un couple `(points, explication)` :
les essais ne portent que sur les points, jamais sur le texte de l'explication.

In [ ]:
def essai(nom, fonction, attendu, tolerance=1e-6):
    try:
        obtenu = fonction()
    except Exception as err:
        print(f"  X   {nom} : {type(err).__name__}: {err}")
        _stats["echec"] += 1
        return

    # None est parfois la BONNE reponse (ex. aucune marque detectee).
    # On ne le lit comme « pas encore fait » que si autre chose etait attendu.
    if obtenu is None and attendu is not None:
        print(f"  ...  {nom} : pas encore fait")
        _stats["attente"] += 1
        return

    if isinstance(attendu, float):
        ok = abs(obtenu - attendu) < tolerance
    else:
        ok = obtenu == attendu

    if ok:
        print(f"  OK   {nom}")
        _stats["ok"] += 1
    else:
        print(f"  X    {nom}   (obtenu {obtenu!r}, attendu {attendu!r})")
        _stats["echec"] += 1


def points(couple):
    """Extrait les points d'un couple (points, explication)."""
    return None if couple is None else couple[0]


def section(titre):
    print(f"\n{titre}")
    print("-" * len(titre))

## Chargement de la table des marques

`table` associe chaque marque à ses domaines officiels, `officiels` est la liste
à plat de tous ces domaines, et `marques` la liste des noms de marques.
Ces trois valeurs servent à presque tous les essais qui suivent.

> Si cette cellule échoue, c'est que `domaines_officiels.csv` n'est pas à côté
> du carnet : vérifiez le dossier de travail avant de continuer.

In [ ]:
table = ph.charger_domaines_officiels()
officiels = ph.tous_les_domaines(table)
marques = list(table.keys())

print("=" * 64)
print("  VERIFICATION DE VOS FONCTIONS")
print("=" * 64)
print(f"{len(table)} marques chargees : {', '.join(marques)}")

---
## Composante 1.1 — extraction et comparaison de domaines

`extraire_domaine_adresse()` et `extraire_domaine_url()`.

In [ ]:
section("Composante 1.1 - extraction et comparaison de domaines")
essai("domaine d'une adresse simple",
      lambda: ph.extraire_domaine_adresse("support123@gmail.com"), "gmail.com")
essai("domaine d'une adresse avec nom",
      lambda: ph.extraire_domaine_adresse("Amazon Support <support123@gmail.com>"),
      "gmail.com")
essai("adresse vide", lambda: ph.extraire_domaine_adresse(""), "")
essai("domaine d'une URL",
      lambda: ph.extraire_domaine_url("https://www.amazon.ca/gp/orders"),
      "www.amazon.ca")
essai("domaine d'une URL piegee",
      lambda: ph.extraire_domaine_url("http://amazon.ca.security-check.xyz/login"),
      "amazon.ca.security-check.xyz")

### Le piège du sous-domaine (courriels n° 6 vs n° 7 du guide)

`accesd.desjardins.com` est un vrai sous-domaine de Desjardins.
`desjardins.com.securite-client.xyz` ne l'est pas : le domaine réel est
`securite-client.xyz`, et « desjardins.com » n'est qu'un préfixe décoratif.

Un test `if "desjardins.com" in domaine` classe le second comme légitime :
c'est exactement l'erreur que l'attaquant provoque.

In [ ]:
section("Composante 1.1 - le piege du sous-domaine (n° 6 vs n° 7)")
essai("accesd.desjardins.com est officiel",
      lambda: ph.domaine_est_officiel("accesd.desjardins.com", officiels), True)
essai("desjardins.com.securite-client.xyz ne l'est PAS",
      lambda: ph.domaine_est_officiel("desjardins.com.securite-client.xyz", officiels),
      False)
essai("amazon.ca est officiel",
      lambda: ph.domaine_est_officiel("amazon.ca", officiels), True)
essai("micr0soft.com ne l'est pas",
      lambda: ph.domaine_est_officiel("micr0soft.com", officiels), False)

### Règles portant sur le domaine d'envoi

Mot d'appât (**+25**), extension à risque (**+10**), substitution de caractères (**+30**).

In [ ]:
section("Composante 1.1 - regles sur le domaine")
essai("mot d'appat present",
      lambda: points(ph.regle_mot_appat("amazon-security-login.com")), 25)
essai("aucun mot d'appat",
      lambda: points(ph.regle_mot_appat("amazon.ca")), 0)
essai("extension a risque",
      lambda: points(ph.regle_extension_risque("netflix-billing.tk")), 10)
essai("extension normale",
      lambda: points(ph.regle_extension_risque("netflix.com")), 0)
essai("substitution 0 pour o",
      lambda: points(ph.regle_substitution("amaz0n.com", marques)), 30)
essai("substitution rn pour m",
      lambda: points(ph.regle_substitution("rnicrosoft.com", marques)), 30)
essai("substitution 1 pour l",
      lambda: points(ph.regle_substitution("paypa1.com", marques)), 30)
essai("domaine sain, aucune substitution",
      lambda: points(ph.regle_substitution("amazon.ca", marques)), 0)

---
## Composante 1.2 — analyse des liens

Deux courriels servent de témoins :

| | `faux_netflix` | `pub` |
|---|---|---|
| nom affiché | Netflix | Super Rabais |
| marque annoncée | `"netflix"` | `None` |

Le second est la raison d'être de `marque_annoncee()` : son lien `bit.ly` est
douteux, mais le courriel ne se fait passer pour personne. La règle du lien
incohérent doit donc rester **MUETTE** — sinon elle signalerait presque tous
les courriels du monde, y compris les légitimes.

In [ ]:
section("Composante 1.2 - analyse des liens")
faux_netflix = {"name": "Netflix", "subject": "Echec de paiement",
                "message": "Mettez a jour vos informations."}
pub = {"name": "Super Rabais", "subject": "LIQUIDATION TOTALE",
       "message": "Achetez maintenant http://bit.ly/aff9921"}
essai("marque annoncee detectee",
      lambda: ph.marque_annoncee(faux_netflix, marques), "netflix")
essai("aucune marque annoncee (pub generique)",
      lambda: ph.marque_annoncee(pub, marques), None)
essai("marque annoncee + lien officiel -> 0",
      lambda: points(ph.regle_lien_incoherent(
          "https://www.netflix.com/browse", "netflix", table)), 0)
essai("marque annoncee + sous-domaine officiel -> 0",
      lambda: points(ph.regle_lien_incoherent(
          "https://accesd.desjardins.com/identification", "desjardins", table)), 0)
essai("marque annoncee + lien piege -> 25",
      lambda: points(ph.regle_lien_incoherent(
          "http://netflix-billing.tk/update.php", "netflix", table)), 25)
essai("PIEGE : aucune marque annoncee -> la regle reste MUETTE",
      lambda: points(ph.regle_lien_incoherent(
          "http://bit.ly/aff9921", None, table)), 0)
essai("adresse IP",
      lambda: points(ph.regle_lien_adresse_ip("http://203.0.113.45/desjardins/login")), 25)
essai("nom de domaine normal",
      lambda: points(ph.regle_lien_adresse_ip("https://www.amazon.ca/")), 0)
essai("raccourcisseur bit.ly",
      lambda: points(ph.regle_lien_raccourcisseur("https://bit.ly/3xK9pQ")), 15)
essai("raccourcisseur tinyurl",
      lambda: points(ph.regle_lien_raccourcisseur("https://tinyurl.com/xk991")), 15)
essai("lien sans https",
      lambda: points(ph.regle_lien_sans_https("http://netflix-billing.tk/update.php")), 10)
essai("lien avec https",
      lambda: points(ph.regle_lien_sans_https("https://www.netflix.com/browse")), 0)

---
## Composante 1.3 — nom affiché vs adresse d'envoi

Le nom affiché est choisi **librement** par l'expéditeur : n'importe qui peut
s'appeler « Amazon Support ». La règle se déclenche seulement si une marque est
citée dans le nom **et** que l'adresse ne lui appartient pas.

« Marie Tremblay » doit rester muet : une règle qui parle sur tous les courriels
ne discrimine rien.

In [ ]:
section("Composante 1.3 - nom affiche vs adresse d'envoi")
essai("marque detectee dans le nom",
      lambda: ph.marque_dans_nom("Amazon Support", marques), "amazon")
essai("aucune marque (nom de personne)",
      lambda: ph.marque_dans_nom("Marie Tremblay", marques), None)
essai("usurpation Amazon via gmail",
      lambda: points(ph.regle_nom_vs_adresse(
          "Amazon Support", "support123@gmail.com", table)), 20)
essai("Desjardins depuis son vrai domaine",
      lambda: points(ph.regle_nom_vs_adresse(
          "AccesD Desjardins", "noreply@desjardins.com", table)), 0)
essai("Microsoft depuis microsoftonline.com",
      lambda: points(ph.regle_nom_vs_adresse(
          "Equipe Microsoft 365", "no-reply@microsoftonline.com", table)), 0)
essai("nom de personne : la regle reste MUETTE",
      lambda: points(ph.regle_nom_vs_adresse(
          "Marie Tremblay", "marie@cegep-exemple.qc.ca", table)), 0)

---
## Composante 1.4 — en-têtes techniques

`Return-Path` (**+15**) et `Reply-To` (**+15**), tous deux comparés **par domaine**.

Le piège de l'infolettre : `info@infolettre-cuisine.com` et
`bounce-4471@infolettre-cuisine.com` sont deux adresses différentes du **même**
domaine. C'est le fonctionnement normal des rebonds.

Et une donnée absente n'est pas une preuve de fraude : en-tête vide → `(0, "")`.

In [ ]:
section("Composante 1.4 - en-tetes techniques")
essai("Return-Path identique",
      lambda: points(ph.regle_return_path(
          "marie@cegep-exemple.qc.ca", "marie@cegep-exemple.qc.ca")), 0)
essai("Return-Path divergent",
      lambda: points(ph.regle_return_path(
          "securite@desjardins.com", "x9921@relais-etranger.info")), 15)
essai("infolettre : adresse differente, MEME domaine",
      lambda: points(ph.regle_return_path(
          "info@infolettre-cuisine.com", "bounce-4471@infolettre-cuisine.com")), 0)
essai("Return-Path absent : ne rien conclure",
      lambda: points(ph.regle_return_path("rh@entreprise-exemple.qc.ca", "")), 0)
essai("Reply-To vide",
      lambda: points(ph.regle_reply_to("no-reply@netflix.com", "")), 0)
essai("Reply-To detourne",
      lambda: points(ph.regle_reply_to(
          "facturation@fournisseur.ca", "paiements@fournisseur-inv.top")), 15)

---
## Composante 2.1 — mots-clés commerciaux

Trois exigences vérifiées ici : la liste compte **au moins 20 termes**, elle est
**bilingue**, et un mot répété ne compte qu'**une** fois (sinon un seul mot
répété ferait exploser le score). Le plafond de **15 points** est également testé.

In [ ]:
section("Composante 2.1 - mots-cles commerciaux")
essai("liste d'au moins 20 termes",
      lambda: len(sp.MOTS_COMMERCIAUX) >= 20, True)
essai("liste bilingue (au moins un mot accentue ou francais)",
      lambda: any(m in sp.MOTS_COMMERCIAUX
                  for m in ["gratuit", "rabais", "gagnez", "offre"]), True)
essai("un mot repete ne compte qu'une fois",
      lambda: points(sp.regle_mots_commerciaux("gratuit GRATUIT Gratuit")), 3)
essai("aucun mot commercial",
      lambda: points(sp.regle_mots_commerciaux(
          "La reunion de lundi est deplacee a 14 h")), 0)
essai("plafond de 15 respecte",
      lambda: points(sp.regle_mots_commerciaux(
          " ".join(sp.MOTS_COMMERCIAUX))) <= 15, True)

---
## Composante 2.2 — excès de majuscules

`ratio_majuscules()` puis `regle_majuscules()` (**+10**).

Les deux derniers essais sont le **piège de la loi des petits nombres** :
`"URGENT: RE: TP3"` a un ratio de 1,00 et est parfaitement légitime.
Sur 10 lettres, une proportion ne veut rien dire.

In [ ]:
section("Composante 2.2 - exces de majuscules")
essai("tout en majuscules", lambda: sp.ratio_majuscules("BONJOUR"), 1.0)
essai("tout en minuscules", lambda: sp.ratio_majuscules("bonjour"), 0.0)
essai("aucune lettre", lambda: sp.ratio_majuscules("123 !!!"), 0.0)
essai("texte long et criard",
      lambda: points(sp.regle_majuscules(
          "FELICITATIONS VOUS AVEZ GAGNE UN IPHONE GRATUIT")), 10)
essai("texte long et normal",
      lambda: points(sp.regle_majuscules(
          "Votre facture PDF est disponible dans votre Espace client BMO.")), 0)
essai("PIEGE : 'URGENT: RE: TP3' est trop court -> 0",
      lambda: points(sp.regle_majuscules("URGENT: RE: TP3")), 0)
essai("PIEGE : 'ATTENTION' est trop court -> 0",
      lambda: points(sp.regle_majuscules("ATTENTION")), 0)

---
## Composante 2.3 — formulation promotionnelle

Un **motif** exige que plusieurs éléments apparaissent ensemble. D'où les deux
pièges : « croissance de 12 % des ventes » et « taux d'intérêt de 5 % » contiennent
un pourcentage, mais aucune promotion. Un motif qui se contente de chercher `%`
les signale à tort.

In [ ]:
section("Composante 2.3 - formulation promotionnelle")
essai("au moins 8 motifs definis", lambda: len(sp.MOTIFS_PROMO) >= 8, True)
essai("pourcentage de rabais",
      lambda: points(sp.regle_formulation_promo("Profitez de -50 % de rabais")), 10)
essai("pourcentage off en anglais",
      lambda: points(sp.regle_formulation_promo("50 % off today only")), 10)
essai("injonction d'achat",
      lambda: points(sp.regle_formulation_promo("Achetez maintenant, stocks limites")), 10)
essai("PIEGE : pourcentage seul (croissance) -> 0",
      lambda: points(sp.regle_formulation_promo(
          "Rapport trimestriel : croissance de 12 % des ventes")), 0)
essai("PIEGE : taux d'interet -> 0",
      lambda: points(sp.regle_formulation_promo(
          "Le taux d'interet de votre pret passe a 5 % le 1er avril")), 0)
essai("phrase neutre -> 0",
      lambda: points(sp.regle_formulation_promo(
          "La reunion aura lieu a 14 h au local B-204")), 0)

---
## Composante 2.4 — signaux complémentaires

Ponctuation excessive (**+5**) et nombre de liens (**+10**).

In [ ]:
section("Composante 2.4 - signaux complementaires")
essai("trois exclamations",
      lambda: points(sp.regle_exclamations("SUPER OFFRE !!!")), 5)
essai("aucune exclamation",
      lambda: points(sp.regle_exclamations("Bonjour, merci.")), 0)
essai("trois liens",
      lambda: points(sp.regle_nombre_liens(
          "http://a.com http://b.com http://c.com")), 10)
essai("un seul lien",
      lambda: points(sp.regle_nombre_liens("http://a.com")), 0)

---
## Bilan

Le décompte porte sur **tous les essais exécutés depuis le chargement du carnet**.
Si vous n'avez lancé qu'une section, le total sera partiel : relancez tout le
carnet (**Exécution → Tout exécuter**) pour un bilan complet.

In [ ]:
total = _stats["ok"] + _stats["echec"] + _stats["attente"]
print("=" * 64)
print(f"  BILAN : {_stats['ok']} reussis · {_stats['echec']} echoues · "
      f"{_stats['attente']} a faire   (sur {total})")
print("=" * 64)
if _stats["echec"] == 0 and _stats["attente"] == 0:
    print("  Tout est vert. Lancez maintenant :  main.ipynb")
elif _stats["attente"]:
    print("  Continuez : completez les fonctions marquees TODO.")
else:
    print("  Corrigez les X avant de passer a main.ipynb.")
print()